# Lab Feature Store — Feast + Iceberg + Nessie · Jour 2

**Profil docker :** `mlops` · **Durée :** 90 min · **Format :** Trio rôlé

---

## Architecture Feature Store sur la stack existante

```
Silver Iceberg (MinIO/Polaris)
         │
         ▼  Feature engineering (Spark/dbt)
   Feature Store (Feast)
    ├── Offline store : Parquet sur MinIO  ←── Entraînement ML
    └── Online store  : Redis              ←── Inférence temps réel

Nessie branch feat/exp-* ──→ Isolation des expériences ML
```

## Contexte — Incident Feature Skew

> L'équipe ML de RetailCo a entraîné un modèle de churn avec des features calculées
> sur les données Silver du 15 janvier. En production, le modèle sert des features
> calculées sur les données du jour — **feature skew détecté** (+22% d'écart sur recency_days).
>
> Votre mission : reproduire l'incident, le diagnostiquer via Nessie, et corriger
> le pipeline de matérialisation pour garantir la cohérence offline/online.

In [3]:
import os, json, boto3, time, requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from urllib.parse import quote
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ── JARs Iceberg / Nessie / S3A (volume spark-jars monté dans /opt/spark-jars)
JARS = (
    "/opt/spark-jars/iceberg-spark-runtime-3.5_2.12-1.11.0.jar,"
    "/opt/spark-jars/iceberg-aws-bundle-1.11.0.jar,"
    "/opt/spark-jars/nessie-spark-extensions-3.5_2.12-0.105.3.jar,"
    "/opt/spark-jars/hadoop-aws-3.3.4.jar,"
    "/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
)

# ── SparkSession ──────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("RetailCo-FeatureStore")
    .config("spark.jars", JARS)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")
    # Catalog Polaris (REST)
    .config("spark.sql.catalog.polaris",              "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.polaris.catalog-impl", "org.apache.iceberg.rest.RESTCatalog")
    .config("spark.sql.catalog.polaris.uri",          os.getenv("POLARIS_URI", "http://polaris:8181/api/catalog"))
    .config("spark.sql.catalog.polaris.warehouse",    "retailco")         # nom du catalog Polaris
    .config("spark.sql.catalog.polaris.credential",   os.getenv("POLARIS_CREDENTIAL", "root:s3cr3t"))
    .config("spark.sql.catalog.polaris.scope",        "PRINCIPAL_ROLE:ALL")
    # S3FileIO config MinIO — requis par ResolvingFileIO (AWS SDK v2)
    .config("spark.sql.catalog.polaris.s3.endpoint",          os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.sql.catalog.polaris.s3.access-key-id",     os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.sql.catalog.polaris.s3.secret-access-key", os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.sql.catalog.polaris.s3.path-style-access",  "true")
    .config("spark.sql.catalog.nessie.s3.endpoint",           os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.sql.catalog.nessie.s3.access-key-id",      os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.sql.catalog.nessie.s3.secret-access-key",  os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.sql.catalog.nessie.s3.path-style-access",   "true")
    # HadoopFileIO — évite ResolvingFileIO qui requiert AWS SDK v2
    # Source : iceberg.apache.org/docs/latest/aws/#s3-file-io
    # Catalog Nessie
    .config("spark.sql.catalog.nessie",              "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.uri",          os.getenv("NESSIE_URI_V1", "http://nessie:19120/api/v1"))
    .config("spark.sql.catalog.nessie.ref",          "main")
    .config("spark.sql.catalog.nessie.warehouse",    "s3a://warehouse/nessie/")
    # MinIO / S3A
    .config("spark.hadoop.fs.s3a.endpoint",             os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.hadoop.fs.s3a.access.key",           os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.hadoop.fs.s3a.secret.key",           os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.hadoop.fs.s3a.path.style.access",    "true")
    .config("spark.hadoop.fs.s3a.impl",                 "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.defaultCatalog", "polaris")
    .getOrCreate()
)

print(f"✅ Spark {spark.version}")
print(f"✅ Catalog par défaut : polaris")
print(f"✅ Iceberg JARs chargés depuis /opt/spark-jars/")

# ── Clients ───────────────────────────────────────────────────────────────────
NESSIE_API = os.getenv("NESSIE_URI", "http://nessie:19120/api/v2")
MINIO      = os.getenv("MINIO_ENDPOINT", "http://minio:9000")

def nessie_ref(branch_name):
    """Encode les '/' dans le nom de branche pour l'URL Nessie API v2."""
    return quote(branch_name, safe='')


def nessie(method, path, data=None, ignore_conflict=False):
    """Client HTTP Nessie API v2 robuste (gère 204 No Content et erreurs HTTP)."""
    r = requests.request(method, f"{NESSIE_API}{path}",
        headers={"Content-Type": "application/json"}, json=data)
    if not r.text.strip():
        return {}
    try:
        result = r.json()
    except Exception:
        raise RuntimeError(f"Nessie {method} {path} → HTTP {r.status_code} | body non-JSON : {r.text[:300]!r}")
    if r.status_code == 409 and ignore_conflict:
        return result
    if r.status_code >= 400:
        raise RuntimeError(f"Nessie {method} {path} → HTTP {r.status_code} | {result}")
    return result

from feast import FeatureStore
from feast.types import Float32, Float64, Int64, String
print("✅ Feast importé")

✅ Spark 3.5.0
✅ Catalog par défaut : polaris
✅ Iceberg JARs chargés depuis /opt/spark-jars/
✅ Feast importé


---
## ⏱️ Temps 1 — Kata d'amorce (10 min)

Générer les features RFM RetailCo et les matérialiser dans Feast.

In [4]:
# ── Générer les features client depuis Silver ─────────────────────────────────
np.random.seed(42)
n = 500

df_features = pd.DataFrame({
    'customer_id': [f'cust-{i:04d}' for i in range(n)],
    'event_timestamp': [datetime.now() - timedelta(hours=i % 24) for i in range(n)],
    'created_timestamp': [datetime.now()] * n,
    # Features RFM
    'recency_days':    np.random.randint(1, 365, n),
    'frequency_30d':   np.random.randint(0, 15, n),
    'monetary_90d':    np.round(np.random.exponential(50000, n), 2),
    'avg_basket_size': np.round(np.random.uniform(5000, 200000, n), 2),
    'return_rate':     np.round(np.random.beta(2, 8, n), 3),
})

# Sauvegarder comme Parquet pour Feast offline store
import os
os.makedirs('/feast/data', exist_ok=True)
df_features.to_parquet('/feast/data/customer_rfm.parquet', index=False)
print(f'✅ {len(df_features)} features RFM générées')
print(df_features.head(3).to_string())

PermissionError: [Errno 13] Permission denied: '/feast'

In [ ]:
# ── Initialiser Feast et appliquer les définitions ────────────────────────────
os.makedirs('/feast', exist_ok=True)

# Écrire le feature_store.yaml
feast_config = """
project: retailco
provider: local
registry: /feast/registry.db
online_store:
    type: redis
    connection_string: \"redis:6379\"
offline_store:
    type: file
entity_key_serialization_version: 2
"""
with open('/feast/feature_store.yaml', 'w') as f:
    f.write(feast_config)

# Écrire les définitions de features simplifiées
feast_defs = '''
from datetime import timedelta
from feast import Entity, FeatureView, FileSource, Field, ValueType
from feast.types import Float32, Float64, Int64, String

customer = Entity(name="customer_id", join_keys=["customer_id"])

customer_rfm_source = FileSource(
    name="customer_rfm_source",
    path="/feast/data/customer_rfm.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

customer_rfm = FeatureView(
    name="customer_rfm",
    entities=[customer],
    ttl=timedelta(days=30),
    schema=[
        Field(name="recency_days",    dtype=Int64),
        Field(name="frequency_30d",   dtype=Int64),
        Field(name="monetary_90d",    dtype=Float64),
        Field(name="avg_basket_size", dtype=Float64),
        Field(name="return_rate",     dtype=Float32),
    ],
    source=customer_rfm_source,
)
'''
with open('/feast/features.py', 'w') as f:
    f.write(feast_defs)

# Apply
result = subprocess.run(['feast', '-c', '/feast', 'apply'],
    capture_output=True, text=True, cwd='/feast')
print(result.stdout or result.stderr)
print('✅ Feast apply terminé')

---
## ⏱️ Temps 2 — Incident à résoudre (55 min)

### Partie A — Simuler le feature skew (10 min)

In [ ]:
# ── Simuler le feature skew ────────────────────────────────────────────────────
# Scénario : les features d'entraînement ont été calculées sur les données du 15/01
# Les features de production sont calculées sur les données d'aujourd'hui
# → recency_days a dérivé de +22 jours en moyenne

# Features d'entraînement (snapshot J-30)
df_train_features = df_features.copy()
df_train_features['recency_days'] = df_train_features['recency_days'] - 22  # 22 jours de moins
df_train_features['event_timestamp'] = datetime(2024, 1, 15)
df_train_features.to_parquet('/feast/data/customer_rfm_train_snapshot.parquet', index=False)

# Features de production (aujourd'hui)
df_prod_features = df_features.copy()
df_prod_features.to_parquet('/feast/data/customer_rfm.parquet', index=False)

# Mesurer le skew
skew_recency = df_prod_features['recency_days'].mean() - df_train_features['recency_days'].mean()
skew_pct = (df_prod_features['recency_days'].mean() / df_train_features['recency_days'].mean() - 1) * 100

print('📊 Feature Skew détecté :')
print(f'   recency_days (entraînement) : {df_train_features["recency_days"].mean():.1f} jours')
print(f'   recency_days (production)   : {df_prod_features["recency_days"].mean():.1f} jours')
print(f'   Écart absolu                : +{skew_recency:.1f} jours')
print(f'   Écart relatif               : +{skew_pct:.1f} %')
print(f'\n⚠️  Un écart de +{skew_pct:.0f}% sur recency_days dégrade les prédictions de churn.')

### Partie B — Diagnostiquer avec Nessie (15 min)

**TODO 1** : Utilisez Nessie pour retrouver l'état exact des tables Silver au moment de l'entraînement et comparer avec l'état actuel.

In [ ]:
# TODO 1 — Diagnostic via Nessie
# 📝 a) Listez le log Nessie de la branche main
#    b) Identifiez le commit correspondant au 15 janvier
#    c) Lisez la table Silver sur ce hash historique
#    d) Calculez la valeur moyenne de recency_days à cette date
#       et comparez avec aujourd'hui

print('💡 nessie("GET", "/trees/main/history") pour voir les commits')
print('💡 spark.conf.set("spark.sql.catalog.nessie.ref", <hash>) pour pointer sur un commit')
print('💡 Comparer les stats descriptives des features entre les deux états')

In [ ]:
# ✅ SOLUTION TODO 1

# Log Nessie
log = nessie('GET', '/trees/main/history')
print('📋 Log Nessie (commits récents) :')
commits = log.get('logEntries', [])
for c in commits[:5]:
    meta = c.get('commitMeta', {})
    print(f"  {meta.get('hash','')[:12]}  {meta.get('commitTime','')[:19]}  {meta.get('message','')[:40]}")

# Simulation : on sait que le hash correct est v_correct du kata
# En production, on chercherait le hash correspondant au timestamp d'entraînement
print('\n💡 En production :')
print('   1. Récupérer le timestamp d\'entraînement depuis MLflow')
print('   2. Trouver le commit Nessie le plus proche dans le log')
print('   3. Pointer Spark sur ce commit pour reproduire exactement les features d\'entraînement')
print('   4. Comparer les distributions : si delta > seuil → réentraîner')
print(f'\n✅ Diagnostic : skew de +{skew_pct:.0f}% sur recency_days')
print('   Cause : le pipeline de matérialisation ne pointait pas sur le bon état Nessie')

### Partie C — Corriger le pipeline et matérialiser (20 min)

**TODO 2** : Mettez en place un pipeline de matérialisation qui enregistre
le hash Nessie utilisé pour chaque run de matérialisation Feast.

In [ ]:
# TODO 2 — Pipeline de matérialisation avec traçabilité Nessie
# 📝 a) Récupérer le hash Nessie courant
#    b) Calculer les features depuis Silver (avec ce hash)
#    c) Matérialiser dans Feast offline store
#    d) Loguer le hash Nessie avec les métadonnées de matérialisation

print('💡 nessie("GET", "/trees/main")["reference"]["hash"] pour le hash courant')
print('💡 Feast : store.materialize_incremental(end_date=datetime.now())')
print('💡 Stocker le hash dans un fichier de métadonnées ou MLflow tags')

In [ ]:
# ✅ SOLUTION TODO 2 — Pipeline avec traçabilité

def materialize_with_nessie_tracking():
    # 1. Récupérer le hash Nessie courant
    ref = nessie('GET', '/trees/main')
    nessie_hash = ref.get('reference', {}).get('hash', 'unknown')[:16]
    mat_ts = datetime.now()

    print(f'🔄 Démarrage matérialisation')
    print(f'   Nessie hash : {nessie_hash}...')
    print(f'   Timestamp   : {mat_ts.isoformat()}')

    # 2. Calculer les features (ici on réutilise nos données simulées)
    df_features_current = df_prod_features.copy()
    df_features_current['event_timestamp'] = mat_ts
    df_features_current.to_parquet('/feast/data/customer_rfm.parquet', index=False)

    # 3. Matérialiser dans Feast
    try:
        store = FeatureStore(repo_path='/feast')
        store.materialize_incremental(end_date=mat_ts)
        print('✅ Matérialisation Feast terminée (offline → online)')
    except Exception as e:
        print(f'⚠️  Feast materialize: {e}')

    # 4. Traçabilité : sauvegarder les métadonnées
    metadata = {
        'nessie_hash': nessie_hash,
        'materialization_ts': mat_ts.isoformat(),
        'n_customers': len(df_features_current),
        'feature_view': 'customer_rfm',
        'recency_days_mean': round(df_features_current['recency_days'].mean(), 2),
    }
    with open('/feast/data/last_materialization.json', 'w') as f:
        json.dump(metadata, f, indent=2)

    print('\n📋 Métadonnées sauvegardées :')
    print(json.dumps(metadata, indent=2))
    return metadata

meta = materialize_with_nessie_tracking()
print('\n✅ La prochaine fois que le modèle est entraîné, on retrouvera exactement')
print('   quelles features ont été utilisées en cherchant ce hash Nessie dans MLflow.')

---
## ⏱️ Temps 3 — Extension (20 min)

Brancher une expérience ML sur Nessie : créer une branche de features pour tester une nouvelle définition sans affecter la production.

In [ ]:
# ── Expérience sur branche Nessie ─────────────────────────────────────────────
main_hash = nessie('GET', '/trees/main')['reference']['hash']
EXP_BRANCH = 'feat/new-recency-feature'
resp = nessie('POST', f'/trees?name={nessie_ref(EXP_BRANCH)}&type=BRANCH', {
    'type': 'BRANCH',
    'name': 'main',
    'hash': main_hash
}, ignore_conflict=True)
if resp.get('errorCode') == 'REFERENCE_ALREADY_EXISTS':
    print(f'✅ Branche {EXP_BRANCH} déjà existante (relance du notebook) — réutilisée')

# Nouvelle définition de recency : utiliser 14j au lieu de 30j
df_exp_features = df_features.copy()
df_exp_features['recency_days_14d'] = np.random.randint(0, 14, len(df_features))  # nouvelle feature
df_exp_features.to_parquet('/feast/data/customer_rfm_exp.parquet', index=False)

exp_hash = nessie('GET', f'/trees/{nessie_ref(EXP_BRANCH)}')['reference']['hash'][:12]
print(f'✅ Expérience sur branche {EXP_BRANCH} (hash: {exp_hash}...)')
print('\n💡 En production :')
print('   • La branche main continue à servir les features stables en production')
print('   • La branche feat/ est utilisée pour l\'expérience MLflow parallèle')
print('   • Si l\'expérience est concluante, on merge la définition sur main')
print('   • Si elle échoue, on supprime la branche — zéro impact production')

---
## ⏱️ Débrief client (10 min)

### Questions — rôle Client :

1. **"Comment vous garantissez que les features d'entraînement et de production sont les mêmes ? C'est quoi le mécanisme technique ?"**
2. **"Si mon modèle performe moins bien en production, comment vous rétrospectez sur les features qui ont changé ?"**
3. **"Feast + Redis vs acheter une plateforme MLOps complète comme Databricks ou Vertex AI — comment vous justifiez le choix open-source ?"**
4. **"Le Feature Store ajoute une couche de complexité. Pour un client qui commence en ML, vous le recommandez dès le départ ?"**

---

## Récapitulatif

| Concept | Exercice | Impact prod |
|---------|---------|-------------|
| Feature engineering | Features RFM depuis Silver | Calcul centralisé, une seule définition |
| Matérialisation Feast | Offline → Online (Redis) | Inférence temps réel < 10ms |
| Traçabilité Nessie | Hash dans les métadonnées | Reproduire exactement les features d'un modèle |
| Feature skew | Diagnostic + correction | Fiabilité du modèle en production |
| Expérience sur branche | Branch feat/ = sandbox | Tester sans affecter la production |